# Unsway — Phase 4 causal steering

This notebook only restores versioned artifacts and calls repository CLI stages. Select a GPU runtime before running.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print("CUDA device:", torch.cuda.get_device_name(0))

## Clone and install

In [ ]:
import os
import subprocess
from pathlib import Path


def run(*command: str) -> None:
    subprocess.run(command, check=True)

repo = Path("/content/Unsway")
if not repo.exists():
    run("git", "clone", "https://github.com/idris404/Unsway.git", str(repo))
os.chdir(repo)
run("git", "pull", "--ff-only")
run("git", "rev-parse", "HEAD")
run("pip", "-q", "install", "uv")
run("uv", "sync", "--extra", "dev")

## Mount Drive and restore the Phase 3 SAE

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
drive_dir = Path("/content/drive/MyDrive/Unsway/phase3")
os.environ["UNSWAY_DRIVE_DIR"] = str(drive_dir)
Path("data/processed/phase3").mkdir(parents=True, exist_ok=True)
Path("reports").mkdir(exist_ok=True)
run("rsync", "-a", f"{drive_dir}/data/", "data/processed/phase3/")
run("rsync", "-a", f"{drive_dir}/reports/", "reports/")
assert Path("data/processed/phase3/sae.safetensors").is_file()

## Rebuild the ignored Phase 1–2 behavioral inputs

In [ ]:
run("uv", "run", "unsway-phase1", "--config", "configs/phase1.yaml")
run("uv", "run", "unsway-phase2", "--config", "configs/phase2.yaml")

## Validation sweep

This stage selects strengths without reading test outcomes.

In [ ]:
run("uv", "run", "unsway-phase4", "--config", "configs/phase4.yaml", "--stage", "sweep")
phase4_data = drive_dir / "phase4_data"
phase4_reports = drive_dir / "phase4_reports"
phase4_data.mkdir(exist_ok=True)
phase4_reports.mkdir(exist_ok=True)
run("rsync", "-a", "data/processed/phase4/", f"{phase4_data}/")
run("cp", "reports/phase4_validation.json", f"{phase4_reports}/")

## Frozen held-out test

Run only after the validation report has been written. The selected strengths are not changed here.

In [ ]:
run("uv", "run", "unsway-phase4", "--config", "configs/phase4.yaml", "--stage", "test")
run("rsync", "-a", "data/processed/phase4/", f"{phase4_data}/")
run("cp", "reports/phase4_test.json", f"{phase4_reports}/")

## Inspect the causal result

In [ ]:
import json
from pprint import pprint

validation = json.loads(Path("reports/phase4_validation.json").read_text())
test = json.loads(Path("reports/phase4_test.json").read_text())
print("Frozen strengths:")
pprint(validation["selected_strengths"])
print("\nHeld-out test baseline:")
pprint(test["baseline"])
print("\nHeld-out causal interventions:")
for condition in test["conditions"]:
    pprint(
        {
            "intervention": condition["intervention"],
            "strength": condition["strength"],
            "metrics": condition["metrics"],
            "delta": condition["delta"],
        }
    )